In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Algoritmos de Classificação
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# --- PASSO 1: DIVISÃO DOS DADOS (HOLDOUT 60% / 20% / 20%) ---
X_class, y_class = make_classification(n_samples=1000, n_features=10, random_state=42)

# Total (100%) -> Treino Geral (80%) e Teste (20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_class, y_class, test_size=0.20, random_state=42, stratify=y_class
)

# Treino Geral (80%) -> Treino (60%) e Validação (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

# Escalonamento dos dados (sem vazamento de dados)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

def get_class_metrics(y_true, y_pred):
    return {
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, average='weighted'), 4),
        "Recall": round(recall_score(y_true, y_pred, average='weighted'), 4),
        "F1-Score": round(f1_score(y_true, y_pred, average='weighted'), 4)
    }

# Configuração dos hiperparâmetros para busca (Passo 5)
models_class = {
    "KNN": {
        "class": KNeighborsClassifier,
        "params": [{"n_neighbors": k} for k in [3, 5, 7, 9, 15]]
    },
    "Decision Tree": {
        "class": DecisionTreeClassifier,
        "params": [{"max_depth": d, "random_state": 42} for d in [2, 5, 10, None]]
    },
    "Random Forest": {
        "class": RandomForestClassifier,
        "params": [{"n_estimators": n, "max_depth": d, "random_state": 42} for n in [20, 50] for d in [3, 5, 10]]
    },
    "Logistic Regression": {
        "class": LogisticRegression,
        "params": [{"C": c, "random_state": 42} for c in [0.01, 0.1, 1.0, 10.0]]
    }
}

res_train_c, res_val_c, res_test_c = [], [], []

for name, config in models_class.items():
    # --- PASSOS 2, 3 e 4: TREINAMENTO E MEDIÇÃO COM PARÂMETRO DEFAULT ---
    default_model = config["class"]()
    default_model.fit(X_train_scaled, y_train)
    
    # Passo 3: Performance no Treino (Default)
    train_metrics_def = get_class_metrics(y_train, default_model.predict(X_train_scaled))
    res_train_c.append({"Algoritmo": name, **train_metrics_def})
    
    # Passo 4: Performance na Validação (Default)
    val_metrics_def = get_class_metrics(y_val, default_model.predict(X_val_scaled))
    res_val_c.append({"Algoritmo": name, **val_metrics_def})
    
    # --- PASSO 5: BUSCA PELOS MELHORES PARÂMETROS PARA CONTROLAR OVERFITTING ---
    best_score = -1
    best_params = None
    
    for p in config["params"]:
        model = config["class"](**p)
        model.fit(X_train_scaled, y_train)
        y_val_pred = model.predict(X_val_scaled)
        score = accuracy_score(y_val, y_val_pred)
        
        if score > best_score:
            best_score = score
            best_params = p
            
    # --- PASSOS 6 e 7: UNIR TREINO + VALIDAÇÃO E RETREINAR (MODELO LAST) ---
    final_model = config["class"](**best_params)
    final_model.fit(X_train_full_scaled, y_train_full)
    
    # --- PASSO 8: MEDIR PERFORMANCE NO TESTE (GENERALIZAÇÃO) ---
    test_metrics = get_class_metrics(y_test, final_model.predict(X_test_scaled))
    res_test_c.append({"Algoritmo": name, **test_metrics})

# Exibição das Tabelas do Ensaio
print("=== 1) CLASSIFICAÇÃO: DADOS DE TREINO (DEFAULT) ===")
print(pd.DataFrame(res_train_c).to_string(index=False))

print("\n=== 2) CLASSIFICAÇÃO: DADOS DE VALIDAÇÃO (DEFAULT) ===")
print(pd.DataFrame(res_val_c).to_string(index=False))

print("\n=== 3) CLASSIFICAÇÃO: DADOS DE TESTE (MODELO OTIMIZADO) ===")
print(pd.DataFrame(res_test_c).to_string(index=False))

=== 1) CLASSIFICAÇÃO: DADOS DE TREINO (DEFAULT) ===
          Algoritmo  Accuracy  Precision  Recall  F1-Score
                KNN    0.8650     0.8664  0.8650    0.8649
      Decision Tree    1.0000     1.0000  1.0000    1.0000
      Random Forest    1.0000     1.0000  1.0000    1.0000
Logistic Regression    0.8717     0.8717  0.8717    0.8717

=== 2) CLASSIFICAÇÃO: DADOS DE VALIDAÇÃO (DEFAULT) ===
          Algoritmo  Accuracy  Precision  Recall  F1-Score
                KNN     0.820     0.8212   0.820    0.8198
      Decision Tree     0.895     0.8954   0.895    0.8950
      Random Forest     0.930     0.9302   0.930    0.9300
Logistic Regression     0.880     0.8800   0.880    0.8800

=== 3) CLASSIFICAÇÃO: DADOS DE TESTE (MODELO OTIMIZADO) ===
          Algoritmo  Accuracy  Precision  Recall  F1-Score
                KNN     0.855     0.8611   0.855    0.8544
      Decision Tree     0.895     0.8982   0.895    0.8948
      Random Forest     0.890     0.8939   0.890    0.8897
Logis